# Klasifikasi Kategori Produk Indonesia

Model neural network sederhana (PyTorch) untuk memprediksi kategori produk berdasarkan nama produknya.

**Pipeline:**
1. Load dataset dari GitHub
2. Split train/test
3. TF-IDF vectorization (analog StandardScaler tapi untuk teks)
4. Label encoding
5. Konversi ke tensor PyTorch
6. Definisi model MLP
7. Training
8. Evaluasi
9. Inference pada nama produk baru
10. Save model + preprocessors

In [8]:
import torch
import torch.nn as nn
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import LabelEncoder

torch.manual_seed(42)

In [9]:
URL = "https://raw.githubusercontent.com/imadegautama/ml-product-category/refs/heads/main/products_dataset.csv"

df = pd.read_csv(URL)

X = df['product_name'].values
y = df['label'].values

print(f"Total data: {len(df)}")
print(f"\nDistribusi kategori:")
print(df['label'].value_counts())
df.head()

Total data: 5000

Distribusi kategori:
label
Olahraga                  625
Kesehatan & Kecantikan    625
Elektronik                625
Minuman                   625
Fashion                   625
Makanan                   625
Rumah Tangga              625
Buku & Alat Tulis         625
Name: count, dtype: int64


,product_name,label
0,Push Up Bar Oranye XL Terbaru,Olahraga
1,Lipstik Matte Isi 24,Kesehatan & Kecantikan
2,Pemanggang Roti Kuning Isi 10,Elektronik
3,Papan Ski,Olahraga
4,Baterai AA Putih 1 Lusin,Elektronik


In [10]:
# Split train/test dengan stratify biar distribusi kelas seimbang di kedua set
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (4000,)
X_test shape: (1000,)


In [11]:
# TF-IDF: ubah nama produk jadi vektor numerik
# ngram_range=(1,2) artinya pakai unigram (kata tunggal) + bigram (pasangan kata)
# min_df=2 artinya buang kata yang muncul cuma di 1 dokumen
vectorizer = TfidfVectorizer(
    lowercase=True,
    ngram_range=(1, 2),
    min_df=2,
)

# Fit di train, transform di kedua-duanya (sama kayak StandardScaler)
X_train_vec = vectorizer.fit_transform(X_train).toarray()
X_test_vec = vectorizer.transform(X_test).toarray()

print(f"Jumlah fitur (vocab size): {X_train_vec.shape[1]}")
print(f"X_train_vec shape: {X_train_vec.shape}")
print(f"\nContoh 5 kata pertama di vocab: {vectorizer.get_feature_names_out()[:5]}")

Jumlah fitur (vocab size): 1791
X_train_vec shape: (4000, 1791)

Contoh 5 kata pertama di vocab: ['10' '10 mini' '10000mah' '100g' '100g jumbo']


In [12]:
# Label encoding: ubah label string jadi integer (0, 1, 2, ...)
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

print(f"Jumlah kelas: {len(label_encoder.classes_)}")
print(f"Mapping kelas:")
for i, cls in enumerate(label_encoder.classes_):
    print(f"  {i}: {cls}")

Jumlah kelas: 8
Mapping kelas:
  0: Buku & Alat Tulis
  1: Elektronik
  2: Fashion
  3: Kesehatan & Kecantikan
  4: Makanan
  5: Minuman
  6: Olahraga
  7: Rumah Tangga


In [13]:
# Konversi ke tensor PyTorch
X_train_tensor = torch.tensor(X_train_vec, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_vec, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_enc, dtype=torch.long)
y_test_tensor = torch.tensor(y_test_enc, dtype=torch.long)

print(f"X_train_tensor shape: {X_train_tensor.shape}")
print(f"y_train_tensor shape: {y_train_tensor.shape}")

X_train_tensor shape: torch.Size([4000, 1791])
y_train_tensor shape: torch.Size([4000])


In [14]:
class ProductClassifier(nn.Module):
    def __init__(self, input_size, hidden_size=128, output_size=8, dropout=0.3):
        super(ProductClassifier, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)  # cegah overfitting
        self.fc2 = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# Instantiate model — input size mengikuti vocab size dari TF-IDF
input_size = X_train_tensor.shape[1]
num_classes = len(label_encoder.classes_)

model = ProductClassifier(
    input_size=input_size,
    hidden_size=128,
    output_size=num_classes
)
print(model)

ProductClassifier(
  (fc1): Linear(in_features=1791, out_features=128, bias=True)
  (relu): ReLU()
  (dropout): Dropout(p=0.3, inplace=False)
  (fc2): Linear(in_features=128, out_features=8, bias=True)
)


In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

epochs = 100
for epoch in range(epochs):
    model.train()  # mode training (dropout aktif)

    # Forward pass
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    # Backward and optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}')

Epoch [10/100], Loss: 0.9369
Epoch [20/100], Loss: 0.0367
Epoch [30/100], Loss: 0.0038
Epoch [40/100], Loss: 0.0017
Epoch [50/100], Loss: 0.0014
Epoch [60/100], Loss: 0.0012
Epoch [70/100], Loss: 0.0014
Epoch [80/100], Loss: 0.0011
Epoch [90/100], Loss: 0.0009
Epoch [100/100], Loss: 0.0010


In [ ]:
from sklearn.metrics import classification_report, accuracy_score

# Set ke evaluation mode (dropout nonaktif)
model.eval()

with torch.no_grad():
    test_outputs = model(X_test_tensor)
    _, predictions = torch.max(test_outputs, 1)

y_pred = predictions.numpy()
y_true = y_test_tensor.numpy()

accuracy = accuracy_score(y_true, y_pred)
print(f'Test Accuracy: {accuracy * 100:.2f}%\n')
print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=label_encoder.classes_))

## Inference: Prediksi Nama Produk Baru

Bagian ini buat ngetes model dengan nama produk yang belum pernah dilihat (termasuk yang gak persis ada di training data).

In [16]:
def predict(product_names):
    """Prediksi kategori untuk satu atau banyak nama produk."""
    if isinstance(product_names, str):
        product_names = [product_names]

    # Vectorize pakai vectorizer yang sama (sudah di-fit di training)
    vec = vectorizer.transform(product_names).toarray()
    tensor = torch.tensor(vec, dtype=torch.float32)

    model.eval()
    with torch.no_grad():
        outputs = model(tensor)
        probs = torch.softmax(outputs, dim=1)
        _, preds = torch.max(outputs, 1)

    results = []
    for name, pred_idx, prob in zip(product_names, preds, probs):
        label = label_encoder.inverse_transform([pred_idx.item()])[0]
        confidence = prob[pred_idx].item() * 100
        results.append({
            'product': name,
            'predicted_category': label,
            'confidence': f'{confidence:.2f}%'
        })
    return results


# Test pada produk-produk baru
test_products = [
    "Cilok Bandung Pedas",
    "Es Kopi Susu Gula Aren",
    "Hoodie Oversize Hitam",
    "Laptop Gaming RTX 4060",
    "Sunscreen SPF 50 PA+++",
    "Sapu Lidi",
    "Raket Tenis Wilson",
    "Buku Tulis 100 Lembar",
    # Produk yang gak persis ada di training - test generalisasi
    "Pisang Goreng Madu",
    "iPhone 15 Pro Max",
]

results = predict(test_products)
for r in results:
    print(f"{r['product']:<35} -> {r['predicted_category']:<25} ({r['confidence']})")

Cilok Bandung Pedas                 -> Makanan                   (100.00%)
Es Kopi Susu Gula Aren              -> Minuman                   (100.00%)
Hoodie Oversize Hitam               -> Fashion                   (99.97%)
Laptop Gaming RTX 4060              -> Elektronik                (99.99%)
Sunscreen SPF 50 PA+++              -> Kesehatan & Kecantikan    (100.00%)
Sapu Lidi                           -> Rumah Tangga              (99.94%)
Raket Tenis Wilson                  -> Olahraga                  (100.00%)
Buku Tulis 100 Lembar               -> Buku & Alat Tulis         (100.00%)
Pisang Goreng Madu                  -> Makanan                   (100.00%)
iPhone 15 Pro Max                   -> Elektronik                (99.96%)


## Save Model + Preprocessors

Buat inference nanti, kita butuh **3 file**:
- `product_classifier.pth` — bobot model
- `vectorizer.pkl` — TF-IDF vectorizer (untuk transform input baru)
- `label_encoder.pkl` — label encoder (untuk decode prediksi jadi nama kategori)

In [17]:
import pickle
from google.colab import files

# Save model weights
torch.save(model.state_dict(), 'product_classifier.pth')

# Save vectorizer & label encoder (dibutuhkan buat inference)
with open('vectorizer.pkl', 'wb') as f:
    pickle.dump(vectorizer, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(label_encoder, f)

print("Saved:")
print("  - product_classifier.pth")
print("  - vectorizer.pkl")
print("  - label_encoder.pkl")

# Download ke local
files.download('product_classifier.pth')
files.download('vectorizer.pkl')
files.download('label_encoder.pkl')

Saved:
  - product_classifier.pth
  - vectorizer.pkl
  - label_encoder.pkl


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>